# GridLock Hackathon 2.0 — Traffic Demand Prediction
## ML Pipeline v5 — day-48 train / day-49 val / day-49 trajectory features

**Data structure:**  
- Train day 48: all 24 hours (~69k rows)  
- Train day 49: hours 0–2 only (~7.8k rows)  
- Test: day 49, hours 2–13 (41,778 rows)  

**Strategy:** train on day 48 only, validate on day 49.  
All target encodings computed from day 48 rows only.  
Day 49 trajectory features (hours 0/1/2 demand per geohash) provide a momentum signal.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stop, log_evaluation as lgb_log
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import pygeohash as pgh

plt.style.use('ggplot')
pd.set_option('display.max_columns', 80)
np.random.seed(42)
print('Imports successful.')

## Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Train day counts: {train["day"].value_counts().sort_index().to_dict()}')
print(f'Test  day counts: {test["day"].value_counts().sort_index().to_dict()}')

---
## Step 1: Feature Engineering

All stats (imputation, frequencies, target encodings) come from **day 48 only**.  
These are then applied uniformly to day 48 train, day 49 val, and test rows.

In [ ]:
class FeatureEngineer:
    'v5 — fits stats on day 48 rows only; transforms any df with those stats.'

    @staticmethod
    def _decode_gh(gh):
        try:
            lat, lng, lat_err, lng_err = pgh.decode_exactly(gh)
            return float(lat), float(lng)
        except Exception:
            return 0.0, 0.0

    @staticmethod
    def _parse_ts(ts):
        h, m = ts.split(':')
        return int(h), int(m)

    @staticmethod
    def _time_of_day(h):
        if   h <= 5:  return 0
        elif h <= 11: return 1
        elif h <= 17: return 2
        else:         return 3

    def fit(self, df):
        'Fit ALL stats on day 48 rows only.'
        d48 = df[df['day'] == 48].copy()

        # Imputation stats (day 48 only)
        self._temp_med_global = float(d48['Temperature'].median())
        self._temp_med_gh     = d48.groupby('geohash')['Temperature'].median()
        self._rt_mode_global  = (d48['RoadType'].mode().iloc[0]
                                 if len(d48['RoadType'].mode()) else 'Residential')
        self._rt_mode_gh      = d48.groupby('geohash')['RoadType'].apply(
            lambda x: x.mode().iloc[0] if len(x.mode()) else self._rt_mode_global)
        self._wx_mode_global  = (d48['Weather'].mode().iloc[0]
                                 if len(d48['Weather'].mode()) else 'Sunny')
        self._wx_mode_gh      = d48.groupby('geohash')['Weather'].apply(
            lambda x: x.mode().iloc[0] if len(x.mode()) else self._wx_mode_global)

        # Original-NaN trackers (computed at transform time per-df)

        # Categorical levels (from day 48)
        self._rt_cols = ['Residential', 'Street', 'Highway']
        self._wx_cols = ['Sunny', 'Rainy', 'Foggy', 'Snowy']

        # Geohash hierarchy frequencies (day 48 only)
        self._gh_freq  = d48['geohash'].value_counts().astype(float)
        gh4 = d48['geohash'].str[:4]
        gh5 = d48['geohash'].str[:5]
        self._gh4_freq = gh4.value_counts().astype(float)
        self._gh5_freq = gh5.value_counts().astype(float)
        self._gh4_labels = {v: i for i, v in enumerate(sorted(gh4.unique()))}
        self._gh5_labels = {v: i for i, v in enumerate(sorted(gh5.unique()))}

        # Temperature mean per geohash (day 48)
        self._gh_temp_mean = d48.groupby('geohash')['Temperature'].mean()
        self._global_temp_mean = float(d48['Temperature'].mean())

        # Fallback geohash demand mean (day 48) — for unseen-geohash trajectory fills
        self._gh_d48_mean   = d48.groupby('geohash')['demand'].mean()
        self._global_d48_mean = float(d48['demand'].mean())

        self.fitted = True
        return self

    def transform(self, df):
        assert self.fitted, 'Call fit first'
        df = df.copy()

        # Track original NaN before imputation (Unknown indicators)
        df['RoadType_was_nan'] = df['RoadType'].isna().astype(int)
        df['Weather_was_nan']  = df['Weather'].isna().astype(int)

        # Impute with day-48 stats
        df['Temperature'] = df['Temperature'].fillna(
            df['geohash'].map(self._temp_med_gh).fillna(self._temp_med_global))
        df['RoadType'] = df['RoadType'].fillna(
            df['geohash'].map(self._rt_mode_gh).fillna(self._rt_mode_global))
        df['Weather'] = df['Weather'].fillna(
            df['geohash'].map(self._wx_mode_gh).fillna(self._wx_mode_global))

        # Timestamp features
        ts = df['timestamp'].apply(self._parse_ts)
        df['hour']         = ts.apply(lambda x: x[0])
        df['minute']       = ts.apply(lambda x: x[1])
        df['hour_sin']     = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos']     = np.cos(2 * np.pi * df['hour'] / 24)
        df['is_rush_hour'] = df['hour'].isin([7,8,9,17,18,19]).astype(int)
        df['time_of_day']  = df['hour'].apply(self._time_of_day)

        # Geohash decode
        geo = df['geohash'].apply(self._decode_gh)
        df['geohash_lat'] = geo.apply(lambda x: x[0])
        df['geohash_lng'] = geo.apply(lambda x: x[1])

        # Geohash hierarchy strings (for later target encoding keys)
        df['geohash_4'] = df['geohash'].str[:4]
        df['geohash_5'] = df['geohash'].str[:5]
        df['geohash_4_label'] = df['geohash_4'].map(self._gh4_labels).fillna(-1).astype(int)
        df['geohash_5_label'] = df['geohash_5'].map(self._gh5_labels).fillna(-1).astype(int)
        df['geohash_frequency'] = df['geohash'].map(self._gh_freq).fillna(0.0)

        # Road/infra
        df['LargeVehicles'] = (df['LargeVehicles'] == 'Allowed').astype(int)
        df['Landmarks']     = (df['Landmarks']     == 'Yes').astype(int)
        for rt in self._rt_cols:
            df[f'RoadType_{rt}'] = (df['RoadType'] == rt).astype(int)
        for wx in self._wx_cols:
            df[f'Weather_{wx}']  = (df['Weather']  == wx).astype(int)

        # Weather/temp
        df['temp_x_is_rush'] = df['Temperature'] * df['is_rush_hour']
        df['temp_deviation'] = df['Temperature'] - df['geohash'].map(
            self._gh_temp_mean).fillna(self._global_temp_mean)

        return df

print('FeatureEngineer v5 defined (day-48-only stats).')

In [ ]:
fe = FeatureEngineer().fit(train)
train_fe = fe.transform(train)
test_fe  = fe.transform(test)
print(f'Train FE shape: {train_fe.shape}')
print(f'Test  FE shape: {test_fe.shape}')
print(f'Train day counts after FE: {train_fe["day"].value_counts().sort_index().to_dict()}')

---
## Step 2: Target Encodings (day 48 only, α=10 smoothing)

Smoothing formula: `(count * group_mean + α * global_mean) / (count + α)`  
Stats come from day 48 only; applied uniformly to train (both days) and test.

In [ ]:
def encode_d48(train_fe, test_fe, group_cols, col_name,
               agg='mean', target='demand', alpha=10):
    'Compute day-48 group stat (smoothed) and apply to all train + test rows.'
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    d48 = train_fe[train_fe['day'] == 48]
    gm  = float(d48[target].mean()) if agg == 'mean' else float(d48[target].agg(agg))
    stats = d48.groupby(group_cols)[target].agg([agg, 'count'])
    stats.columns = [agg, 'count']
    if agg == 'mean':
        stats['smooth'] = (stats['count']*stats[agg] + alpha*gm) / (stats['count'] + alpha)
        d = stats['smooth'].to_dict()
    else:
        # Don't smooth median/std — they're robustness/variability signals
        d = stats[agg].fillna(gm).to_dict()

    if len(group_cols) == 1:
        tr_keys = train_fe[group_cols[0]].tolist()
        te_keys = test_fe[group_cols[0]].tolist()
    else:
        tr_keys = list(zip(*[train_fe[c].tolist() for c in group_cols]))
        te_keys = list(zip(*[test_fe[c].tolist() for c in group_cols]))
    train_fe[col_name] = [d.get(k, gm) for k in tr_keys]
    test_fe[col_name]  = [d.get(k, gm) for k in te_keys]

print('encode_d48 defined.')

In [ ]:
print('Computing day-48 target encodings...')
encode_d48(train_fe, test_fe, ['geohash', 'hour'],   'geohash_hour_mean',   agg='mean',   alpha=10)
print('  ✓ geohash_hour_mean')
encode_d48(train_fe, test_fe, 'geohash',             'geohash_mean',        agg='mean',   alpha=10)
print('  ✓ geohash_mean')
encode_d48(train_fe, test_fe, ['geohash', 'hour'],   'geohash_hour_median', agg='median')
print('  ✓ geohash_hour_median')
encode_d48(train_fe, test_fe, ['geohash', 'hour'],   'geohash_hour_std',    agg='std')
print('  ✓ geohash_hour_std')
encode_d48(train_fe, test_fe, ['geohash_4', 'hour'], 'geohash_4_hour_mean', agg='mean',   alpha=10)
print('  ✓ geohash_4_hour_mean')
encode_d48(train_fe, test_fe, 'geohash_5',           'geohash_5_mean',      agg='mean',   alpha=10)
print('  ✓ geohash_5_mean')

# Fix std NaN (single-row groups)
d48_demand_std = float(train_fe.loc[train_fe['day']==48, 'demand'].std())
train_fe['geohash_hour_std'] = train_fe['geohash_hour_std'].fillna(d48_demand_std)
test_fe['geohash_hour_std']  = test_fe['geohash_hour_std'].fillna(d48_demand_std)

print(f'\nTrain columns after encoding: {len(train_fe.columns)}')
print('Sample encoded values (day 48 row vs day 49 row):')
sample_cols = ['day','geohash','hour','geohash_hour_mean','geohash_mean','geohash_4_hour_mean','demand']
print(train_fe[sample_cols].iloc[[0, train_fe[train_fe['day']==49].index[0]]].to_string())

---
## Step 3: Day 49 Trajectory Features

Use the day-49 train rows (hours 0/1/2) to compute per-geohash demand momentum.  
Applied uniformly to all train and test rows (geohashes without day-49 data use day-48 fallback).

In [ ]:
d49 = train_fe[train_fe['day'] == 49]

d49_h0 = d49[d49['hour'] == 0].groupby('geohash')['demand'].mean().to_dict()
d49_h1 = d49[d49['hour'] == 1].groupby('geohash')['demand'].mean().to_dict()
d49_h2 = d49[d49['hour'] == 2].groupby('geohash')['demand'].mean().to_dict()

gh_d48_mean   = fe._gh_d48_mean.to_dict()
global_d48    = fe._global_d48_mean

def gh_lookup(g, primary, secondary):
    if g in primary:
        return primary[g]
    return secondary.get(g, global_d48)

for df in [train_fe, test_fe]:
    df['demand_hour0']         = df['geohash'].map(lambda g: gh_lookup(g, d49_h0, gh_d48_mean))
    df['demand_hour1']         = df['geohash'].map(lambda g: gh_lookup(g, d49_h1, gh_d48_mean))
    df['demand_hour2_partial'] = df['geohash'].map(lambda g: gh_lookup(g, d49_h2, gh_d48_mean))
    df['demand_day49_trend']   = df['demand_hour1'] - df['demand_hour0']

print('Trajectory features added.')
print(f'\nDay 49 trajectory coverage:')
print(f'  hour 0: {len(d49_h0)} geohashes ({100*len(d49_h0)/train_fe["geohash"].nunique():.1f}%)')
print(f'  hour 1: {len(d49_h1)} geohashes ({100*len(d49_h1)/train_fe["geohash"].nunique():.1f}%)')
print(f'  hour 2: {len(d49_h2)} geohashes ({100*len(d49_h2)/train_fe["geohash"].nunique():.1f}%)')

print('\nTest stats for trajectory features:')
print(test_fe[['demand_hour0','demand_hour1','demand_hour2_partial','demand_day49_trend']].describe().round(4))

---
## Step 4: Feature Columns

In [ ]:
EXCLUDE = {
    'Index', 'geohash', 'timestamp', 'demand',
    'RoadType', 'Weather',
    'geohash_4', 'geohash_5',
}
feature_cols = [c for c in train_fe.columns if c not in EXCLUDE]
print(f'Total features: {len(feature_cols)}')
for i, c in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {c}')

In [ ]:
X_all = train_fe[feature_cols].values.astype(np.float32)
y_all = train_fe['demand'].values.astype(np.float32)

nan_cnt = int(np.isnan(X_all).sum())
if nan_cnt > 0:
    X_all = np.nan_to_num(X_all, nan=0.0)
    print(f'Replaced {nan_cnt} NaN values with 0')
print(f'X_all: {X_all.shape}  |  y_all: {y_all.shape}')
print(f'y range: [{y_all.min():.4f}, {y_all.max():.4f}]  mean: {y_all.mean():.4f}')

---
## Step 5: Train / Val Split — day 48 train, day 49 val

This split mirrors exactly how the test set was constructed.  
**Note:** trajectory features are derived from day 49 hours 0–2 → for val rows in those hours, trajectory features partially leak the target. Val R² will be optimistic for those rows. The leaderboard score is the ground truth.

In [ ]:
is_d48 = (train_fe['day'] == 48).values
tr_idx = np.where( is_d48)[0]
va_idx = np.where(~is_d48)[0]

X_train, y_train = X_all[tr_idx], y_all[tr_idx]
X_val,   y_val   = X_all[va_idx], y_all[va_idx]

print(f'Train (day 48): {X_train.shape}  ({100*len(tr_idx)/len(X_all):.1f}%)')
print(f'Val   (day 49): {X_val.shape}    ({100*len(va_idx)/len(X_all):.1f}%)')
print(f'Val   demand  : mean={y_val.mean():.4f}  std={y_val.std():.4f}')
print(f'Train demand  : mean={y_train.mean():.4f}  std={y_train.std():.4f}')

---
## Step 6: Baseline LightGBM

In [ ]:
lgbm_base = LGBMRegressor(
    n_estimators=2000, learning_rate=0.05,
    num_leaves=127, max_depth=8, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    n_jobs=-1, random_state=42, verbose=-1
)
lgbm_base.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb_early_stop(50, verbose=False), lgb_log(200)]
)

base_train_r2 = r2_score(y_train, lgbm_base.predict(X_train))
base_val_r2   = r2_score(y_val,   lgbm_base.predict(X_val))

print(f'Baseline LightGBM')
print(f'  Train R² = {base_train_r2:.4f}  |  Val R² = {base_val_r2:.4f}')
print(f'  Gap      = {base_train_r2 - base_val_r2:.4f}')
print(f'  Best iter: {lgbm_base.best_iteration_}')

base_imp = pd.DataFrame({'feature': feature_cols,
                          'importance': lgbm_base.feature_importances_}
                        ).sort_values('importance', ascending=False)
print('\nTop 20 features (baseline):')
print(base_imp.head(20).to_string(index=False))

---
## Step 7: Optuna Tuning — LightGBM (100 trials)

Maximize val R² on day 49 holdout.

In [ ]:
def objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int(  'n_estimators',     500,  3000),
        learning_rate     = trial.suggest_float('learning_rate',    0.01, 0.10, log=True),
        num_leaves        = trial.suggest_int(  'num_leaves',       63,   511),
        max_depth         = trial.suggest_int(  'max_depth',        4,    9),
        min_child_samples = trial.suggest_int(  'min_child_samples',10,   100),
        reg_alpha         = trial.suggest_float('reg_alpha',        1e-3, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda',       1e-3, 10.0, log=True),
        subsample         = trial.suggest_float('subsample',        0.6,  1.0),
        subsample_freq    = 1,
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5,  1.0),
        n_jobs=-1, random_state=42, verbose=-1
    )
    m = LGBMRegressor(**params)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)],
          callbacks=[lgb_early_stop(50, verbose=False)])
    val_r2 = r2_score(y_val, m.predict(X_val))
    train_r2 = r2_score(y_train, m.predict(X_train))
    trial.set_user_attr('train_r2', train_r2)
    trial.set_user_attr('val_r2',   val_r2)
    trial.set_user_attr('best_iter', m.best_iteration_)
    return val_r2

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_trial = study.best_trial
print(f'\nBest trial val R² : {best_trial.value:.6f}  ({max(0, best_trial.value*100):.2f})')
print(f'        train R²   : {best_trial.user_attrs["train_r2"]:.4f}')
print(f'        gap        : {best_trial.user_attrs["train_r2"] - best_trial.value:.4f}')
print(f'        best_iter  : {best_trial.user_attrs["best_iter"]}')
print('Best params:')
for k, v in best_trial.params.items():
    print(f'  {k}: {v}')

---
## Step 8: Tuned LightGBM — Val Check

In [ ]:
best_params = study.best_params.copy()
best_iter   = best_trial.user_attrs['best_iter']

tuned_check = LGBMRegressor(
    **best_params,
    n_jobs=-1, random_state=42, verbose=-1
)
tuned_check.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                callbacks=[lgb_early_stop(50, verbose=False)])

tuned_train_r2 = r2_score(y_train, tuned_check.predict(X_train))
tuned_val_r2   = r2_score(y_val,   tuned_check.predict(X_val))
tuned_best_iter = tuned_check.best_iteration_

print(f'Tuned LightGBM (day 48 → day 49)')
print(f'  Train R² = {tuned_train_r2:.4f}')
print(f'  Val   R² = {tuned_val_r2:.4f}')
print(f'  Gap      = {tuned_train_r2 - tuned_val_r2:.4f}')
print(f'  Score    = {max(0, tuned_val_r2*100):.2f}')
print(f'  Best iter: {tuned_best_iter}')

tuned_imp = pd.DataFrame({'feature': feature_cols,
                           'importance': tuned_check.feature_importances_}
                         ).sort_values('importance', ascending=False)
print('\nTop 15 features (tuned):')
print(tuned_imp.head(15).to_string(index=False))

---
## Step 9: Retrain on Full Train (day 48 + day 49)

n_estimators = best_iter (from early stopping) + 10% buffer.

In [ ]:
final_iters = int(tuned_best_iter * 1.10) + 1
print(f'Retraining on {X_all.shape[0]} rows with {final_iters} trees...')

final_params = best_params.copy()
final_params['n_estimators'] = final_iters

final_model = LGBMRegressor(
    **final_params,
    n_jobs=-1, random_state=42, verbose=-1
)
final_model.fit(X_all, y_all)
print('Final model trained on full data.')

---
## Step 10: Final Predictions + Submission

In [ ]:
X_test = test_fe[feature_cols].values.astype(np.float32)
nan_te = int(np.isnan(X_test).sum())
if nan_te > 0:
    X_test = np.nan_to_num(X_test, nan=0.0)
    print(f'Replaced {nan_te} NaN values in X_test')

max_train_demand = float(train['demand'].max())
final_preds = final_model.predict(X_test)
final_preds = np.clip(final_preds, 0.0, max_train_demand)

print(f'Prediction distribution (final model):')
print(f'  min   : {final_preds.min():.6f}   (train min: {train["demand"].min():.6f})')
print(f'  max   : {final_preds.max():.6f}   (train max: {train["demand"].max():.6f})')
print(f'  mean  : {final_preds.mean():.6f}  (train mean: {train["demand"].mean():.6f})')
print(f'  std   : {final_preds.std():.6f}   (train std : {train["demand"].std():.6f})')
print(f'  zeros : {100*(final_preds == 0).mean():.1f}%')

In [ ]:
submission = pd.DataFrame({'Index': test['Index'], 'demand': final_preds})
assert submission.shape == (41778, 2), f'Shape mismatch: {submission.shape}'
submission.to_csv('submission.csv', index=False)

print(f'submission.csv saved  |  shape: {submission.shape}')
print('\nFirst 5 rows:')
print(submission.head())
print('\nDescribe:')
print(submission['demand'].describe())

---
## Step 11: Validation Report

In [ ]:
print('=' * 65)
print('  GRIDLOCK HACKATHON 2.0 - VALIDATION REPORT (v5)')
print('=' * 65)

print('\nValidation: train=day 48 (69,427 rows)  |  val=day 49 (7,872 rows)')
print(f'  Baseline LightGBM    train={base_train_r2:.4f}  val={base_val_r2:.4f}  '
      f'gap={base_train_r2-base_val_r2:.4f}')
print(f'  Tuned LightGBM       train={tuned_train_r2:.4f}  val={tuned_val_r2:.4f}  '
      f'gap={tuned_train_r2-tuned_val_r2:.4f}  score={max(0,tuned_val_r2*100):.2f}')

print(f'\n--- Top 15 Tuned Feature Importances ---')
for _, row in tuned_imp.head(15).iterrows():
    print(f'  {row["feature"]:<32s} {row["importance"]:.2f}')

print(f'\n--- Final Submission ---')
print(f'  Model      : Tuned LightGBM, retrained on full train (day 48 + day 49)')
print(f'  n_estimators: {final_iters}')
print(f'  Features   : {len(feature_cols)}')
print(f'  Pred stats : min={final_preds.min():.4f}  max={final_preds.max():.4f}  '
      f'mean={final_preds.mean():.4f}  std={final_preds.std():.4f}')
print(f'  Submission : submission.csv  shape={submission.shape}')
print('=' * 65)